# 🔑 Solutions · Lesson 02: Time-Series Data Preparation

> Try every problem yourself first! Peek at hints before peeking here. Run the setup cell, then read each solution **and its explanation**: there is often more than one valid way.

In [1]:
import sys; sys.path.append("..")            # lets us import course_utils.py from the parent folder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.tools.sm_exceptions         # (imported first: statsmodels registers its own warning filters on import)
import warnings; warnings.filterwarnings("ignore")   # keep the notebook readable; in real projects READ your warnings!
from course_utils import *                    # data loaders, metrics, plotting helpers, check()

In [2]:
raw = pd.read_csv(DATA / "maize_daily_raw.csv")
df = raw.drop_duplicates().copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")
daily = df.copy()
med = daily["price"].rolling(7, center=True, min_periods=3).median()
ratio = daily["price"] / med
daily.loc[(ratio > 2) | (ratio < 0.5), "price"] = np.nan
mkt_days = pd.offsets.CustomBusinessDay(weekmask="Mon Tue Wed Thu Fri Sat")
daily_clean = daily.reindex(pd.date_range(daily.index.min(), daily.index.max(), freq=mkt_days))
daily_clean["price"] = daily_clean["price"].ffill()
daily_clean["rainfall_mm"] = daily_clean["rainfall_mm"].fillna(0)

### ✅ Solution 2.1 · Triage the raw file

**Problem recap:** Load `data/maize_daily_raw.csv` into `raw`. Then answer: (a) how many rows does it have, (b) how many are **exact duplicate rows**,
  (c) how many prices are missing? Finally, drop the duplicates and store the remaining row count in `n_clean`.

In [3]:
raw = pd.read_csv(DATA / 'maize_daily_raw.csv')
print('rows:', len(raw), '| duplicates:', raw.duplicated().sum(), '| missing prices:', raw['price'].isna().sum())
n_clean = len(raw.drop_duplicates())
check('rows after removing duplicates', n_clean, 911)

rows: 919 | duplicates: 8 | missing prices: 5
✅ rows after removing duplicates: correct!


**Why this works:** Always *count* the problems before fixing them: it tells you how worried to be. Here: 919 rows, 8 duplicates, 5 missing prices.

### ✅ Solution 2.2 · Catch the decimal-point typos

**Problem recap:** Starting from the de-duplicated, date-indexed DataFrame `df` in the worked example (before any outlier treatment), find the **data-entry outliers** (prices that are ~10× too big or ~10× too small).
  Use the **ratio of each price to its centred 7-day rolling median**; flag ratios above 2 or below 0.5. Store the number found in `n_outliers`,
  and set those prices to `NaN`.

In [4]:
daily_fixed = df.copy()
med = daily_fixed['price'].rolling(7, center=True, min_periods=3).median()
ratio = daily_fixed['price'] / med
mask = (ratio > 2) | (ratio < 0.5)
print(daily_fixed.loc[mask, 'price'])
n_outliers = int(mask.sum())
daily_fixed.loc[mask, 'price'] = np.nan
check('outliers found', n_outliers, 5)

date
2022-07-12      4.42
2023-06-24    400.40
2023-12-13    495.10
2024-05-29    436.70
2024-06-14      4.32
Name: price, dtype: float64
✅ outliers found: correct!


**Why this works:** Why the median and not the mean? The mean of a 7-day window is *itself* dragged up by a 10× typo, hiding it. The median ignores a single wild value. (Note: it's fine to use a *centred* window here because we are cleaning historical data for inspection, but never use centred windows to build model features. See Problem 2.5.)

### ✅ Solution 2.3 · Weekly vs monthly aggregation

**Problem recap:** Using `daily_clean` (price + rainfall, daily index) from the worked example, resample to **monthly** (`'ME'`) taking the **mean price** and the **total rainfall**.
  Which month was the wettest, and how many millimetres fell?

In [5]:
monthly = daily_clean.resample('ME').agg({'price': 'mean', 'rainfall_mm': 'sum'})
wet = monthly['rainfall_mm'].idxmax()
print(f'Wettest month: {wet:%B %Y} with {monthly.loc[wet, "rainfall_mm"]:.1f} mm')
monthly.head()

Wettest month: April 2023 with 124.8 mm


,price,rainfall_mm
2022-01-31,59.945385,13.3
2022-02-28,56.638750,35.6
2022-03-31,54.805556,120.8
2022-04-30,52.140385,81.1
2022-05-31,48.538077,59.8


**Why this works:** Choosing the right aggregation function is a modelling decision: sum for flows (rain, sales volume), mean or last for levels (prices, temperature), max/min for extremes.

### ✅ Solution 2.4 · Build your own lag features

**Problem recap:** For the weekly **tomato** price build a DataFrame `feat` with these columns, **without** using `make_features`:
  `lag_1`, `lag_4`, `lag_52` (price shifted), `roll4` (mean of the previous 4 weeks), and `rain_lag_8` (rainfall 8 weeks ago).
  Then run `leakage_check` to prove it is leak-free.

In [6]:
tomato = load_weekly('tomato')
s = tomato['price']
def f(s):
    return pd.DataFrame({'lag_1': s.shift(1), 'lag_4': s.shift(4), 'lag_52': s.shift(52),
                         'roll4': s.shift(1).rolling(4).mean(),
                         'rain_lag_8': tomato['rainfall_mm'].shift(8).reindex(s.index)})
feat = f(s)
print(feat.tail(3).round(2))
leakage_check(f, s)

            lag_1  lag_4  lag_52  roll4  rain_lag_8
date                                               
2024-12-15  54.24  47.13   54.95  44.52        40.9
2024-12-22  46.16  33.24   63.58  44.28        10.3
2024-12-29  58.29  43.47   67.69  50.54        31.7
✅ No leakage detected: past rows never changed when the future was poisoned.


True

**Why this works:** Every feature is a function of data *strictly older* than the row's own timestamp. Note that the rainfall feature is independent of the price series, so poisoning the price can't change it; the real protection is the shift(8).

### ✅ Solution 2.5 · The leakage detective

**Problem recap:** The function `buggy_features` below contains **three different leakage bugs**. Use `leakage_check` to confirm the leak, find the culprit columns,
  then write `fixed_features` that preserves the *idea* of each column but removes the leak.

In [7]:
s = load_weekly('maize')['price']
def buggy_features(s):
    return pd.DataFrame({'smooth': s.rolling(5, center=True).mean(), 'next_wk': s.shift(-1), 'zscore': (s - s.mean()) / s.std()})
leakage_check(buggy_features, s)

def fixed_features(s):
    past = s.shift(1)
    return pd.DataFrame({
        'smooth':  past.rolling(5).mean(),                                  # trailing window on data we already know
        'last_wk': s.shift(1),                                              # lag, not lead
        'zscore':  (past - past.expanding().mean()) / past.expanding().std(),  # statistics from the past only
    })
leakage_check(fixed_features, s)

🚨 LEAKAGE: rows before the 50% cut changed. Culprit column(s): ['smooth', 'next_wk', 'zscore']
✅ No leakage detected: past rows never changed when the future was poisoned.


True

**Why this works:** Leakage hides in *innocent-looking* one-liners. The test is always the same: **would this number have been available at the moment of forecasting?** Global scaling (`StandardScaler` fitted on all data), centred windows, `bfill`, `interpolate` over future gaps and negative shifts all answer 'no'.